# 141 — Computer use basado en visión

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

La solución valida el contrato mínimo sin asumir un valor interno específico.


In [ ]:
result = run_lab("robotics", seed=141)
assert result["kind"] == "robotics"
assert result["evidence"]
show(result)


## Solución 1 — Mapeo de coordenadas

- (a) `sx = 2560/1366 ≈ 1.874`, `sy = 1440/768 = 1.875` (casi iguales: misma
  relación de aspecto).
- (b) `(683·1.874, 384·1.875) ≈ (1280, 720)` — el centro de la pantalla real.
- (c) Sin mapear clicaría en (683, 384) real: a `√((1280−683)² + (720−384)²)
  ≈ 685` píxeles del objetivo — ni siquiera cerca del elemento.
- (d) Si la relación de aspecto cambia, sx ≠ sy y la imagen se deforma: el
  modelo aprende geometrías distorsionadas y el error de grounding deja de
  ser isotrópico. Conservarla mantiene un único factor mental de escala.


In [ ]:
sx, sy = 2560/1366, 1440/768
x, y = 683*sx, 384*sy
d = ((1280-683)**2 + (720-384)**2) ** 0.5
print(f"sx={sx:.3f} sy={sy:.3f} -> real=({x:.0f}, {y:.0f}), error sin mapeo={d:.0f} px")


## Solución 2 — Horizonte con y sin verificación

- (a) p=0.95: 5→77 %, 15→46 %, 40→13 %. p=0.99: 5→95 %, 15→86 %, 40→67 %.
- (b) Con un reintento verificado: `p_paso = 0.95 + 0.05·0.95 = 0.9975`;
  `0.9975⁴⁰ ≈ 90.5 %`. De 13 % a 90 % con un solo reintento por paso.
- (c) La verificación convierte errores *compuestos* (silenciosos, que
  arruinan todo lo posterior) en errores *locales* (detectados y reparados al
  coste de un screenshot). Es la misma lección que DAgger y el cierre de
  lazos: la robustez no viene de no fallar, sino de detectar y corregir.


In [ ]:
for p in (0.95, 0.99):
    print(p, [round(p**n, 2) for n in (5, 15, 40)])
p_paso = 0.95 + 0.05*0.95
print("con reintento:", round(p_paso**40, 3))


## Solución 3 — Postcondiciones

- (a) "El cuadro de búsqueda muestra el texto `informe.pdf`" (visible en el
  campo, no solo tecleado).
- (b) "El contador del carrito incrementó en 1" o "apareció la confirmación
  'añadido al carrito'".
- (c) "El indicador de cambios sin guardar (punto/asterisco del título)
  desapareció".
- (d) "La lista muestra elementos distintos a los del screenshot anterior y
  la barra de scroll bajó".

La más difícil con solo píxeles es **(c)**: el efecto de guardar es un cambio
sutil (a veces invisible) del chrome de la ventana; muchas apps no dan
ninguna señal visual — el caso típico donde conviene una señal adicional
(título de la ventana, o verificación por sistema de archivos si está
permitida).


## Solución 4 — Niveles de riesgo

- **Libre** (reversible, sin efecto externo): abrir pestaña, copiar al
  portapapeles, scroll.
- **Verificar después** (reversible con esfuerzo o efecto local): borrar a la
  papelera (recuperable), cerrar sin guardar (si hay autosave/confirmación de
  la app), enviar formulario de contacto *si* la tarea lo pedía explícitamente
  — con captura previa del contenido como evidencia.
- **Confirmación humana previa** (irreversible o efecto externo/sistémico):
  vaciar la papelera (destrucción definitiva), ejecutar un instalador
  descargado (modifica el sistema y es el vector clásico de malware).

El criterio transversal: **reversibilidad y alcance del efecto**, no la
dificultad técnica del clic — un clic trivial puede ser la acción más
peligrosa de la sesión.


In [ ]:
from ai_evolution.labs import run_lab

result = run_lab("robotics", seed=141)
assert result["kind"] == "robotics" and result["limitations"]
print("evidencias:", len(result["evidence"]))
